![Minha Logo](image_b22985.png)
<img src="../image_b22985.png" width="400">

# Sistema de Recomendação de Veículos: Alternativas de Mercado

### Contexto do Negócio e o Problema de Retenção
Uma plataforma emergente de classificados de veículos enfrenta um desafio de retenção de utilizadores. Atualmente, quando um cliente visita a página de um modelo específico (como um Toyota Corolla 2020), o site exibe uma secção de "carros similares" que mostra apenas outros veículos idênticos. A equipa de Produto identificou que os utilizadores abandonam o site rapidamente quando não encontram o preço ou a quilometragem ideal exatamente naquele modelo procurado.

### A Solução Proposta
O objetivo de negócio é reter esse cliente apresentando alternativas viáveis de mercado através de uma nova secção estratégica: "Outros que pode gostar". O foco principal na construção do MVP deste motor de recomendação é sugerir carros com características e faixas de preço semelhantes, mas com a exigência estrita de apresentar opções de outras marcas e modelos. 

### O Desafio Técnico (Cold Start)
Como a plataforma é nova, não existe um histórico consolidado de navegação, cliques ou compras dos utilizadores — o que caracteriza um problema clássico de "Cold Start". Diante deste cenário, a inteligência do sistema de recomendação será baseada puramente nas características físicas e de mercado dos próprios veículos.

### Dados e Escopo do Desenvolvimento
Considerando o escopo de um Produto Mínimo Viável (MVP), os dados foram extraídos do dataset público "vagnerbessa/average-car-prices-bazil". A arquitetura da solução foi desenhada para resolver os seguintes desafios técnicos:

* **Feature Engineering:** Selecionar as colunas relevantes e traduzir variáveis textuais (como tipo de combustível e caixa de velocidades) para um formato matemático compreensível pelo algoritmo.
* **Escalonamento:** Tratar a grande diferença de escala entre as variáveis, garantindo que grandezas maiores (como o preço) não ofusquem atributos cruciais menores (como o tamanho do motor).
* **Motor Matemático e Regras de Negócio:** Definir a lógica para calcular o nível de similaridade entre os veículos e aplicar regras de negócio rígidas para evitar recomendações com disparidades extremas de preço, garantindo tecnicamente a exibição prioritária de marcas concorrentes.

In [ ]:
import pandas as pd
from IPython.display import display, HTML, clear_output

df = pd.read_csv('fipe_2022.csv')

df_base = df[df['month_of_reference'] == 'January'].copy()

colunas_uteis = ['brand', 'model', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl', 'age_years']
df_carros = df_base[colunas_uteis].copy().reset_index(drop=True)

df_carros['termo_busca'] = (
    df_carros['brand'].astype(str) + " " + 
    df_carros['model'].astype(str) + " " + 
    df_carros['year_model'].astype(str) + " " + 
    df_carros['engine_size'].astype(str)
).str.lower()

print(f"Base de dados carregada com sucesso! Total de veículos únicos: {len(df_carros)}")

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

vars_numericas = ['engine_size', 'year_model', 'avg_price_brl', 'age_years']
vars_categoricas = ['fuel', 'gear']

preparador = ColumnTransformer([
    ('numeros', StandardScaler(), vars_numericas),
    ('categorias', OneHotEncoder(handle_unknown='ignore'), vars_categoricas)
])

matriz_carros = preparador.fit_transform(df_carros)

print("Matriz matemática gerada com sucesso! O algoritmo já entende as características dos veículos.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar_concorrentes_semelhantes(id_carro, top_n=10, margem_preco=0.25):
    vetor_escolhido = matriz_carros[id_carro].reshape(1, -1)
    
    similaridades = cosine_similarity(vetor_escolhido, matriz_carros).flatten()
    
    df_resultado = df_carros.copy()
    df_resultado['score_similaridade'] = similaridades
    
    carro_original = df_carros.iloc[id_carro]
    marca_original = carro_original['brand']
    preco_original = carro_original['avg_price_brl']
    
    filtro_outras_marcas = df_resultado['brand'] != marca_original
    
    preco_minimo = preco_original * (1 - margem_preco)
    preco_maximo = preco_original * (1 + margem_preco)
    filtro_preco_justo = df_resultado['avg_price_brl'].between(preco_minimo, preco_maximo)
    
    recomendacoes = df_resultado[filtro_outras_marcas & filtro_preco_justo].sort_values(
        by='score_similaridade', ascending=False
    )
    
    return carro_original, recomendacoes.head(top_n)

In [ ]:
print("=" * 70)
print(" BEM-VINDO AO CONSULTOR AUTOMOTIVO SAUTER DIGITAL ")
print("=" * 70)

pesquisa = input("Digite a marca, modelo, ano ou motor do carro que você gosta (ex: 'toyota corolla 2020'): ").lower()

termos = pesquisa.split()
resultados_busca = df_carros.copy()

for termo in termos:
    resultados_busca = resultados_busca[resultados_busca['termo_busca'].str.contains(termo, na=False)]

if resultados_busca.empty:
    print(f"\n Nenhum carro encontrado com os termos: '{pesquisa}'. Tente ser mais genérico.")
else:
    clear_output(wait=True)
    display(HTML("<h3> Encontramos estes modelos. Digite o NÚMERO do carro exato que você quer analisar:</h3>"))
    
    opcoes_exibicao = resultados_busca.head(15)
    for i, (idx, row) in enumerate(opcoes_exibicao.iterrows()):
        print(f" [{i}] {row['brand']} {row['model']} ({row['year_model']}) - Motor: {row['engine_size']} - Preço: R$ {row['avg_price_brl']:,.2f}")
    
    try:
        escolha_idx = int(input("\n Seu número escolhido: "))
        id_real_do_carro = opcoes_exibicao.index[escolha_idx]
        
        carro_alvo, concorrentes = buscar_concorrentes_semelhantes(id_real_do_carro, top_n=10)
        
        clear_output(wait=True)
        
        painel_html = f"""
        <div style="background-color: #2c3e50; padding: 20px; border-radius: 10px; color: white; font-family: Arial;">
            <h2 style="margin-top:0; color: #f1c40f;"> SEU CARRO DE INTERESSE</h2>
            <p style="font-size: 18px; margin: 5px 0;"><b>{carro_alvo['brand']} - {carro_alvo['model']} ({carro_alvo['year_model']})</b></p>
            <hr style="border-color: #34495e;">
            <div style="display: flex; gap: 40px; font-size: 15px; margin-top: 10px;">
                <span>💰 <b>Preço:</b> R$ {carro_alvo['avg_price_brl']:,.2f}</span>
                <span>⚙️ <b>Motor:</b> {carro_alvo['engine_size']}</span>
                <span>⛽ <b>Combustível:</b> {carro_alvo['fuel']}</span>
            </div>
        </div>
        <h3 style="color: #2c3e50; margin-top: 30px;">🔥 Alternativas de OUTRAS MARCAS com características semelhantes:</h3>
        """
        display(HTML(painel_html))
        
        tabela_final = concorrentes[['brand', 'model', 'year_model', 'engine_size', 'fuel', 'avg_price_brl', 'score_similaridade']].copy()
        tabela_final.columns = ['Marca', 'Modelo', 'Ano', 'Motor', 'Combustível', 'Preço (R$)', 'Similaridade']
        
        tabela_estilizada = tabela_final.style.format({
            'Preço (R$)': 'R$ {:,.2f}',
            'Similaridade': '{:.1%}'
        }).bar(subset=['Similaridade'], color='#27ae60', vmin=0, vmax=1)\
          .hide(axis='index')\
          .set_properties(**{'text-align': 'left', 'padding': '10px', 'border-bottom': '1px solid #ddd'})
          
        display(tabela_estilizada)

    except (ValueError, IndexError):
        print("\n Número inválido. Por favor, rode a célula novamente e digite um número válido da lista.")